<a href="https://colab.research.google.com/github/kiryu-arai/kaggle_compedition_monster/blob/suzuki/0520_kaggle_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# kaggle APIのインストール
!pip install kaggle

In [15]:
# driveのマウント
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [16]:
#パイプライン構成
import os
import json

f = open("kaggle.json", 'r') # ディレクトリは必要に応じて変更してください
json_data = json.load(f)
os.environ['KAGGLE_USERNAME'] = json_data['username']
os.environ['KAGGLE_KEY'] = json_data['key']

In [17]:
# データのダウンロードと解凍
!kaggle competitions download -c ambl-california-housing
!unzip /content/ambl-california-housing.zip

ambl-california-housing.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/ambl-california-housing.zip
  inflating: sample.csv              
  inflating: test.csv                
  inflating: train.csv               


In [18]:
#データの移動
import os
import shutil

destination_folder = '/content/drive/MyDrive/kaggle_data'

# フォルダが存在しない場合は作成
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)
    print(f"フォルダ '{destination_folder}' を作成しました。")
else:
    print(f"フォルダ '{destination_folder}' は既に存在します。")

files_to_move = ['sample.csv', 'test.csv', 'train.csv']

for file_name in files_to_move:
    source_path = os.path.join('/content/', file_name)
    destination_path = os.path.join(destination_folder, file_name)
    if os.path.exists(source_path):
        shutil.move(source_path, destination_path)
        print(f"'{file_name}' を '{destination_folder}' に移動しました。")
    else:
        print(f"'{file_name}' は存在しませんでした。")

フォルダ '/content/drive/MyDrive/kaggle_data' は既に存在します。
'sample.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'test.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'train.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。


### モジュールの準備

In [19]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# 前処理(正規化・標準化)
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# データ分割
from sklearn.model_selection import train_test_split

# 線形モデル
from sklearn.linear_model import LinearRegression

# 精度評価
from sklearn.metrics import mean_squared_error

# グラフをアウトプット行に出力するためのマジックコマンド
%matplotlib inline

### データセットの読み込み

In [20]:
train = pd.read_csv('/content/drive/MyDrive/kaggle_data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/kaggle_data/test.csv')
sample = pd.read_csv('/content/drive/MyDrive/kaggle_data/sample.csv')

### 今回利用するデータ
- 今回使うデータセットのカラムは以下になります。

|カラム名|意味|
|-|-|
|MedInc|ブロックの所得中央値|
|HouseAge|ブロックの家屋年齢の中央値|
|AveRooms|1世帯当たりの平均居室数|
|AveBedrms|1世帯当たりの平均寝室数|
|Population|ブロックの人口|
|AveOccup|平均世帯人口|
|Latitude|緯度|
|Longitude|経度|
|**Price**|住宅価格(目的変数)|






### データの説明

- 1990年の米国国勢調査から得られたデータ
- 一定のエリア(ブロックグループ)で集計されている(1ブロックグループ辺り大体600人から3000人)
- レコード数は20640で欠損値はなし
- 目的変数はカルフォルニア州の住宅価格の中央値で10万ドル単位で表示している
  - 例) 4.3 → 43万ドル

In [ ]:
# データの表示
train.head()

In [ ]:
test.head()

In [ ]:
sample.head()

In [ ]:
# 分布状況の可視化
train.hist(bins=30)
plt.tight_layout()

In [ ]:
# 分布状況の可視化
test.hist(bins=30)
plt.tight_layout()

In [ ]:
# 相関係数
cor = test.corr()

# ヒートマップ
sns.heatmap(cor, annot=True)
plt.show()

### 特徴量作成

In [21]:
# 世帯数(Household)を求める
train['Household'] = train['Population'] / train['AveOccup']
test['Household'] = test['Population'] / test['AveOccup']

# 合計居室数(AllRooms)と合計寝室数(AllBedrms)を求める
train["AllRooms"] = train["Household"] * train["AveRooms"]
train["AllBedrms"] = train["Household"] * train["AveBedrms"]
test["AllRooms"] = test["Household"] * test["AveRooms"]
test["AllBedrms"] = test["Household"] * test["AveBedrms"]

### データ分割

In [22]:
# データを説明変数と目的変数に分ける
train_X = train.drop(['Price', 'id'], axis=1)
target = train['Price']

print('データ分割前の行数' ,len(train_X))

データ分割前の行数 16512


In [23]:
# データの分割(学習用データ80% 検証用データ 20%)
# test_sizeで検証用データの割合を設定することができる

X_train, X_test, y_train, y_test = train_test_split(train_X, target, test_size=0.2, random_state=2)

# 分割後のデータ数を確認
print('学習用データの行数' ,len(X_train))
print('検証用用データの行数' ,len(X_test))

学習用データの行数 13209
検証用用データの行数 3303


### 特徴量選択

In [24]:
# REF法を利用した特徴量選択（変数選択）
from sklearn.feature_selection import RFE

# インスタンス
rfe = RFE(LinearRegression())

# 特徴量選定（変数選択）の実施
X_rfe = rfe.fit(X_train,y_train)
X_selected = X_train.columns[X_rfe.support_]

# 選択した特徴量
print(X_selected)

Index(['MedInc', 'AveRooms', 'AveBedrms', 'Latitude', 'Longitude'], dtype='object')


### モデルの学習

In [25]:
# xgboostのimport
from xgboost import XGBRFRegressor

# xgboostのインスタンス化
XGBoost = XGBRFRegressor()

# 学習の実行
model = XGBoost.fit(X_train[X_selected], y_train)

# モデルの予測 学習済みモデルを使用して、新しいデータに対して予測を行うために、predict関数を利用できます。
pred_xg_train = model.predict(X_train[X_selected])
pred_xg_test = model.predict(X_test[X_selected])

# 数値の丸め込み処理 0以下を0に5.00001以上を5.00001に変換する
pred_xg_train_rounding = np.where(pred_xg_train <= 0, 0, pred_xg_train)
pred_xg_train_rounding = np.where(pred_xg_train_rounding >= 5.00001, 5.00001, pred_xg_train_rounding)

pred_xg_test_rounding = np.where(pred_xg_test <= 0, 0, pred_xg_test)
pred_xg_test_rounding = np.where(pred_xg_test_rounding >= 5.00001, 5.00001, pred_xg_test_rounding)

### 精度評価

In [26]:
# RMSE RMSEはMSEに√を付けたものになるのでnp.sqrt()でMSEを囲んで算出する
print('RMSE_train',np.sqrt(mean_squared_error(y_train, pred_xg_train_rounding)))
print('RMSE_test',np.sqrt(mean_squared_error(y_test, pred_xg_test_rounding)))

RMSE_train 0.6594998267027213
RMSE_test 0.6695383803173565


### アンサンブル学習の導入

アンサンブル学習とは、複数の機械学習モデルを組み合わせて、より良い予測結果を得る手法です。ここでは、`XGBoost`、`RandomForestRegressor`、`GradientBoostingRegressor`を組み合わせた`VotingRegressor`を使用します。


In [27]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from xgboost import XGBRFRegressor

# ベースとなるモデルを定義
# XGBoostRegressor はすでに定義済みですが、ここでは新しいインスタンスを作成します。
# random_state を設定して再現性を確保します。
xgb_model = XGBRFRegressor(random_state=42)
rf_model = RandomForestRegressor(random_state=42)
gbr_model = GradientBoostingRegressor(random_state=42)

# VotingRegressor を作成
# 重み (weights) を調整して、各モデルの貢献度を変えることができます。
# ここでは同じ重みを与えています。
ensemble_model = VotingRegressor(
    estimators=[
        ('xgb', xgb_model),
        ('rf', rf_model),
        ('gbr', gbr_model)
    ],
    weights=[1, 1, 1] # 各モデルの重み
)

# アンサンブルモデルの学習
print("アンサンブルモデルの学習を開始します...")
ensemble_model.fit(X_train[X_selected], y_train)
print("アンサンブルモデルの学習が完了しました。")

アンサンブルモデルの学習を開始します...
アンサンブルモデルの学習が完了しました。


### アンサンブルモデルによる精度評価

In [28]:
# アンサンブルモデルで予測
pred_ensemble_train = ensemble_model.predict(X_train[X_selected])
pred_ensemble_test = ensemble_model.predict(X_test[X_selected])

# 数値の丸め込み処理
pred_ensemble_train_rounding = np.where(pred_ensemble_train <= 0, 0, pred_ensemble_train)
pred_ensemble_train_rounding = np.where(pred_ensemble_train_rounding >= 5.00001, 5.00001, pred_ensemble_train_rounding)

pred_ensemble_test_rounding = np.where(pred_ensemble_test <= 0, 0, pred_ensemble_test)
pred_ensemble_test_rounding = np.where(pred_ensemble_test_rounding >= 5.00001, 5.00001, pred_ensemble_test_rounding)

# RMSEを計算
print('Ensemble RMSE_train',np.sqrt(mean_squared_error(y_train, pred_ensemble_train_rounding)))
print('Ensemble RMSE_test',np.sqrt(mean_squared_error(y_test, pred_ensemble_test_rounding)))

Ensemble RMSE_train 0.44145593897718466
Ensemble RMSE_test 0.547478880731252


### アンサンブルモデルによる提出用ファイルの作成

In [29]:
# 全学習データを使って最終的なアンサンブルモデルを学習
print("最終アンサンブルモデルの学習を開始します...")
ensemble_model.fit(train_X[X_selected], target)
print("最終アンサンブルモデルの学習が完了しました。")

# テストデータで予測
pred_ensemble_final = ensemble_model.predict(test[X_selected])

# 数値の丸め込み処理
pred_ensemble_final_rounding = np.where(pred_ensemble_final <= 0, 0, pred_ensemble_final)
pred_ensemble_final_rounding = np.where(pred_ensemble_final_rounding >= 5.00001, 5.00001, pred_ensemble_final_rounding)

# sampleに予測値を代入
sample_ensemble = sample.copy()
sample_ensemble['Price'] = pred_ensemble_final_rounding

# 提出用CSVファイルの作成
sample_ensemble.to_csv('/content/drive/MyDrive/kaggle_data/submit_ensemble.csv', index=None)

print("提出用ファイル 'submit_ensemble.csv' を作成しました。")
display(sample_ensemble.head())

最終アンサンブルモデルの学習を開始します...
最終アンサンブルモデルの学習が完了しました。
提出用ファイル 'submit_ensemble.csv' を作成しました。


,id,Price
0,0,3.136108
1,1,1.596039
2,2,1.105912
3,3,3.199383
4,4,4.046145


In [30]:
# 作成したファイルをKaggleに直接投稿
!kaggle competitions submit -c ambl-california-housing -f /content/drive/MyDrive/kaggle_data/submit_ensemble.csv -m "Ensemble Learning Submission via Colab"

100% 94.1k/94.1k [00:00<00:00, 476kB/s]
Successfully submitted to AMBL初心者向けコンペティション_California Housing